# LeetCode 1230 — Toss Strange Coins bằng Python

Link đề: https://leetcode.com/problems/toss-strange-coins/ · Độ khó: **Medium** · Chủ đề: Array, Math, Dynamic Programming, Probability and Statistics

> **Bài Premium — đề viết theo hiểu biết về bài, nên đối chiếu lại nếu có tài khoản Premium.** Đề và chữ ký hàm được đối chiếu với bản mô tả công khai tại https://leetcode.doocs.org/en/lc/1230/ (method `probabilityOfHeads(prob, target)`).

Skeleton LeetCode:

```python
class Solution:
    def probabilityOfHeads(self, prob: List[float], target: int) -> float:
        ...
```

Ghi chú: trả về **xác suất** (số thực) để số đồng xu ngửa đúng bằng `target`; chấp nhận sai số `10^-5`.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu bài toán **phân phối Poisson–nhị thức** (Poisson binomial): tung `n` đồng xu **độc lập** với xác suất ngửa **khác nhau**, hỏi xác suất có đúng `k` đồng ngửa.
2. Xây được **DP xác suất** `f[i][j]` = xác suất trong `i` đồng đầu có đúng `j` đồng ngửa, với chuyển trạng thái "đồng thứ `i` ngửa hoặc sấp".
3. Đi từ **liệt kê `2^n` kết quả** → **đệ quy có nhớ** → **DP bảng 2 chiều** → **DP 1 chiều (rolling array)** duyệt `j` **giảm dần**.
4. Nhìn bài toán dưới góc **nhân đa thức** (hàm sinh — generating function): `Π (1 − p_i + p_i·x)`, hệ số của `x^target` là đáp án.
5. Tránh các lỗi: duyệt `j` tăng dần trong DP 1 chiều, dùng công thức nhị thức `C(n,k)p^k(1−p)^(n−k)` khi xác suất khác nhau, quên cập nhật trường hợp `j = 0`, đệ quy quá sâu.

## 2. Đề bài & ví dụ

Bạn có một số đồng xu. Đồng xu thứ `i` có xác suất `prob[i]` ra **mặt ngửa** (heads) khi tung.

Trả về xác suất số đồng xu ra mặt ngửa **đúng bằng** `target` nếu tung **mỗi** đồng xu đúng một lần.

**Ví dụ 1:** `prob = [0.4], target = 1` → `0.40000`.

**Ví dụ 2:** `prob = [0.5,0.5,0.5,0.5,0.5], target = 0` → `0.03125` (`= 0.5^5`).

**Ràng buộc:**
- `1 <= prob.length <= 1000`
- `0 <= prob[i] <= 1`
- `0 <= target <= prob.length`
- Đáp án được chấp nhận nếu sai lệch không quá `10^-5`.

In [1]:
import math
import random
import time
from fractions import Fraction
from functools import lru_cache
from itertools import product
from typing import List

examples = [
    ([0.4], 1, 0.4),
    ([0.5, 0.5, 0.5, 0.5, 0.5], 0, 0.03125),
]
for prob, target, want in examples:
    print(f"prob={prob}, target={target} -> {want:.5f}")

prob=[0.4], target=1 -> 0.40000
prob=[0.5, 0.5, 0.5, 0.5, 0.5], target=0 -> 0.03125


## 3. Kiến thức nền

### 3.1 Biến cố độc lập và quy tắc nhân

Các lần tung **độc lập** nên xác suất của một kết quả cụ thể (ví dụ đồng 1 ngửa, đồng 2 sấp) bằng **tích**: `p1 · (1 − p2)`.

In [2]:
p = [0.4, 0.7]
print("ngửa-ngửa:", p[0] * p[1])
print("ngửa-sấp :", p[0] * (1 - p[1]))
print("sấp-ngửa :", (1 - p[0]) * p[1])
print("sấp-sấp  :", (1 - p[0]) * (1 - p[1]))
print("tổng     :", p[0] * p[1] + p[0] * (1 - p[1]) + (1 - p[0]) * p[1] + (1 - p[0]) * (1 - p[1]))

ngửa-ngửa: 0.27999999999999997
ngửa-sấp : 0.12000000000000002
sấp-ngửa : 0.42
sấp-sấp  : 0.18000000000000002
tổng     : 1.0


### 3.2 Quy tắc cộng cho các kết quả rời nhau

"Đúng 1 đồng ngửa" gồm hai kết quả rời nhau `(ngửa, sấp)` và `(sấp, ngửa)` → cộng xác suất: `0.4·0.3 + 0.6·0.7 = 0.54`.

In [3]:
print("P(đúng 1 ngửa) =", 0.4 * 0.3 + 0.6 * 0.7)

P(đúng 1 ngửa) = 0.54


### 3.3 `itertools.product` — liệt kê mọi tổ hợp kết quả

`product([0, 1], repeat=n)` sinh mọi dãy `n` bit — mỗi dãy là một kết quả tung (1 = ngửa). Có `2^n` dãy.

In [4]:
for outcome in product([0, 1], repeat=3):
    print(outcome, "số ngửa =", sum(outcome))

(0, 0, 0) số ngửa = 0
(0, 0, 1) số ngửa = 1
(0, 1, 0) số ngửa = 1
(0, 1, 1) số ngửa = 2
(1, 0, 0) số ngửa = 1
(1, 0, 1) số ngửa = 2
(1, 1, 0) số ngửa = 2
(1, 1, 1) số ngửa = 3


### 3.4 So sánh số thực với sai số cho phép

Số thực có sai số làm tròn, nên so sánh bằng `abs(a − b) < eps` hoặc `math.isclose`. Để kiểm chứng **chính xác**, dùng `Fraction` (xác suất `0.4` viết thành `Fraction(2, 5)`).

In [5]:
a = 0.1 * 3
print(a, a == 0.3, abs(a - 0.3) < 1e-9, math.isclose(a, 0.3))
print(Fraction(1, 10) * 3 == Fraction(3, 10))

0.30000000000000004 False True True
True


## 4. Ý tưởng nền tảng / trực giác

Tính tay với `prob = [0.4, 0.7, 0.5]`, `target = 2`. Bảng "sau khi tung `i` đồng, xác suất có `j` đồng ngửa":

| i \ j | 0 | 1 | 2 | 3 |
|---|---|---|---|---|
| 0 (chưa tung) | 1 | 0 | 0 | 0 |
| 1 (p=0.4) | 0.6 | 0.4 | 0 | 0 |
| 2 (p=0.7) | 0.6·0.3 = 0.18 | 0.6·0.7 + 0.4·0.3 = 0.54 | 0.4·0.7 = 0.28 | 0 |
| 3 (p=0.5) | 0.09 | 0.18·0.5 + 0.54·0.5 = 0.36 | 0.54·0.5 + 0.28·0.5 = 0.41 | 0.14 |

Đáp án `f[3][2] = 0.41`. Mỗi ô chỉ phụ thuộc **hai ô ở hàng trên**: "đã có `j` ngửa và đồng mới sấp" hoặc "đã có `j−1` ngửa và đồng mới ngửa". Đó chính là DP.

In [6]:
p = [0.4, 0.7, 0.5]
row = [1.0, 0.0, 0.0, 0.0]
print("i=0:", row)
for i, pi in enumerate(p, 1):
    row = [row[j] * (1 - pi) + (row[j - 1] * pi if j > 0 else 0) for j in range(4)]
    print(f"i={i}:", [round(v, 4) for v in row])

i=0: [1.0, 0.0, 0.0, 0.0]
i=1: [0.6, 0.4, 0.0, 0.0]
i=2: [0.18, 0.54, 0.28, 0.0]
i=3: [0.09, 0.36, 0.41, 0.14]


## 5. Cách 1 — Brute Force / baseline

**Ý tưởng:** liệt kê mọi kết quả `2^n`, tính xác suất từng kết quả bằng quy tắc nhân, cộng những kết quả có đúng `target` mặt ngửa.

In [7]:
def heads_brute(prob, target):
    total = 0.0
    for outcome in product([0, 1], repeat=len(prob)):
        if sum(outcome) != target:
            continue
        pr = 1.0
        for p, h in zip(prob, outcome):
            pr *= p if h else 1 - p
        total += pr
    return total


def heads_brute_exact(prob, target):
    """Như trên nhưng prob là list Fraction -> kết quả chính xác."""
    total = Fraction(0)
    for outcome in product([0, 1], repeat=len(prob)):
        if sum(outcome) == target:
            pr = Fraction(1)
            for p, h in zip(prob, outcome):
                pr *= p if h else 1 - p
            total += pr
    return total


for prob, target, want in examples:
    got = heads_brute(prob, target)
    print(f"{got:.5f}")
    assert abs(got - want) < 1e-9
print("chính xác:", heads_brute_exact([Fraction(2, 5), Fraction(7, 10), Fraction(1, 2)], 2))

0.40000
0.03125
chính xác: 41/100


**Độ phức tạp:** Time `O(2^n · n)`, Space `O(n)`. Với `n = 1000` thì `2^1000` — không tưởng.

**Nút thắt:** nhiều kết quả có **chung tiền tố** (ví dụ 3 đồng đầu giống nhau) nhưng ta tính lại tích từ đầu. Thông tin duy nhất cần nhớ về tiền tố là **đã có bao nhiêu mặt ngửa** — không cần biết mặt nào.

## 6. Bước đột phá

**Trạng thái:** `f[i][j]` = xác suất trong `i` đồng đầu tiên có **đúng** `j` đồng ngửa.

**Chuyển trạng thái** (xét đồng thứ `i`, xác suất ngửa `p = prob[i−1]`):
- **Nếu** đồng `i` **sấp** (xác suất `1 − p`) **thì** `j` ngửa phải đến từ `i − 1` đồng đầu: `f[i−1][j]·(1 − p)`.
- **Nếu** đồng `i` **ngửa** (xác suất `p`) **thì** `i − 1` đồng đầu có `j − 1` ngửa: `f[i−1][j−1]·p` (chỉ khi `j ≥ 1`).

`f[i][j] = f[i−1][j]·(1 − p) + f[i−1][j−1]·p`, với `f[0][0] = 1`, `f[0][j>0] = 0`. Đáp án `f[n][target]`.

**Tối ưu bộ nhớ:** hàng `i` chỉ cần hàng `i − 1` → dùng **một mảng**. Nhưng phải duyệt `j` **từ lớn xuống nhỏ**: khi cập nhật `f[j]` ta cần `f[j−1]` **cũ** (hàng trước); duyệt giảm dần đảm bảo `f[j−1]` chưa bị ghi đè. (Giống knapsack 0/1.)

**Cắt bớt:** chỉ cần `j ≤ target`, vì số ngửa vượt `target` không bao giờ giảm lại.

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Đệ quy có nhớ (top-down)

`go(i, k)` = xác suất trong các đồng `i, i+1, ..., n−1` có đúng `k` đồng ngửa.

In [8]:
def heads_memo(prob, target):
    n = len(prob)

    @lru_cache(maxsize=None)
    def go(i, k):
        if k < 0 or k > n - i:          # cần nhiều ngửa hơn số đồng còn lại -> 0
            return 0.0
        if i == n:
            return 1.0 if k == 0 else 0.0
        p = prob[i]
        return p * go(i + 1, k - 1) + (1 - p) * go(i + 1, k)

    return go(0, target)


for prob, target, want in examples:
    assert abs(heads_memo(prob, target) - want) < 1e-9
print("Cách 2 đúng trên ví dụ")

Cách 2 đúng trên ví dụ


**Độ phức tạp:** Time `O(n · target)` trạng thái, mỗi trạng thái `O(1)`. Space `O(n · target)` cho cache **và độ sâu đệ quy `n`** — với `n = 1000` sát giới hạn đệ quy mặc định của Python (~1000) → dễ `RecursionError`. Vì vậy ta chuyển sang bottom-up.

### 7.2 Cách 3 — DP bảng 2 chiều (bottom-up)

In [9]:
def heads_dp2d(prob, target):
    n = len(prob)
    f = [[0.0] * (target + 1) for _ in range(n + 1)]
    f[0][0] = 1.0
    for i in range(1, n + 1):
        p = prob[i - 1]
        for j in range(0, min(i, target) + 1):
            f[i][j] = f[i - 1][j] * (1 - p)
            if j > 0:
                f[i][j] += f[i - 1][j - 1] * p
    return f[n][target]


for prob, target, want in examples:
    assert abs(heads_dp2d(prob, target) - want) < 1e-9
print("Cách 3 đúng trên ví dụ:", heads_dp2d([0.4, 0.7, 0.5], 2))

Cách 3 đúng trên ví dụ: 0.41000000000000003


**Độ phức tạp:** Time `O(n · target)` ≤ `10^6`, Space `O(n · target)`.

### 7.3 Cách 4 — DP một chiều, duyệt `j` giảm dần

In [10]:
def heads_dp1d(prob, target):
    f = [0.0] * (target + 1)
    f[0] = 1.0
    for p in prob:
        for j in range(target, 0, -1):          # GIẢM dần: f[j-1] vẫn là giá trị cũ
            f[j] = f[j] * (1 - p) + f[j - 1] * p
        f[0] *= 1 - p                            # j = 0: chỉ có trường hợp sấp
    return f[target]


for prob, target, want in examples:
    assert abs(heads_dp1d(prob, target) - want) < 1e-9
print("Cách 4 đúng trên ví dụ")

Cách 4 đúng trên ví dụ


**Độ phức tạp:** Time `O(n · target)`, Space `O(target)`.

### 7.4 Cách 5 — Góc nhìn nhân đa thức (hàm sinh)

Mỗi đồng xu ứng với đa thức `(1 − p_i) + p_i·x` (hệ số của `x^0` là xác suất sấp, `x^1` là xác suất ngửa). Tung độc lập ⇔ **nhân** các đa thức. Hệ số của `x^k` trong tích = xác suất có đúng `k` ngửa. DP ở Cách 3/4 thực chất là nhân lần lượt từng đa thức bậc 1 vào tích.

In [11]:
def poly_mul(a, b):
    out = [0.0] * (len(a) + len(b) - 1)
    for i, x in enumerate(a):
        for j, y in enumerate(b):
            out[i + j] += x * y
    return out


def heads_poly(prob, target):
    poly = [1.0]
    for p in prob:
        poly = poly_mul(poly, [1 - p, p])
    return poly[target]


full = [1.0]
for p in [0.4, 0.7, 0.5]:
    full = poly_mul(full, [1 - p, p])
print("toàn bộ phân phối:", [round(v, 4) for v in full], "| tổng =", round(sum(full), 10))
for prob, target, want in examples:
    assert abs(heads_poly(prob, target) - want) < 1e-9

toàn bộ phân phối: [0.09, 0.36, 0.41, 0.14] | tổng = 1.0


**Độ phức tạp:** Time `O(n²)` (đa thức bậc tăng dần tới `n`, không cắt ở `target`), Space `O(n)`. Lợi ích: có **cả phân phối** một lần — trả lời mọi `target` trong `O(1)`. (Với `n` rất lớn có thể nhân đa thức kiểu chia để trị + FFT, ngoài phạm vi bài.)

## 8. TRACE từng bước

Trace Cách 4 (mảng một chiều) với `prob = [0.4, 0.7, 0.5]`, `target = 2`:

| Sau đồng | p | f[0] | f[1] | f[2] |
|---|---|---|---|---|
| (đầu) | | 1 | 0 | 0 |
| 1 | 0.4 | 0.6 | 0.4 | 0 |
| 2 | 0.7 | 0.18 | 0.54 | 0.28 |
| 3 | 0.5 | 0.09 | 0.36 | **0.41** |

Hàm `trace_dp1d` in cả **thứ tự cập nhật** bên trong mỗi vòng để thấy vì sao phải giảm dần:

In [12]:
def trace_dp1d(prob, target):
    f = [0.0] * (target + 1)
    f[0] = 1.0
    print("ban đầu:", f)
    for i, p in enumerate(prob, 1):
        for j in range(target, 0, -1):
            old_j, old_jm1 = f[j], f[j - 1]
            f[j] = f[j] * (1 - p) + f[j - 1] * p
            print(f"  đồng {i} (p={p}): f[{j}] = {old_j:.4f}*{1 - p:.1f} + {old_jm1:.4f}*{p} = {f[j]:.4f}")
        f[0] *= 1 - p
        print(f"  đồng {i}: f[0] *= {1 - p:.1f} -> {f[0]:.4f} | f = {[round(v, 4) for v in f]}")
    return f[target]


print("Kết quả:", round(trace_dp1d([0.4, 0.7, 0.5], 2), 6))

ban đầu: [1.0, 0.0, 0.0]
  đồng 1 (p=0.4): f[2] = 0.0000*0.6 + 0.0000*0.4 = 0.0000
  đồng 1 (p=0.4): f[1] = 0.0000*0.6 + 1.0000*0.4 = 0.4000
  đồng 1: f[0] *= 0.6 -> 0.6000 | f = [0.6, 0.4, 0.0]
  đồng 2 (p=0.7): f[2] = 0.0000*0.3 + 0.4000*0.7 = 0.2800
  đồng 2 (p=0.7): f[1] = 0.4000*0.3 + 0.6000*0.7 = 0.5400
  đồng 2: f[0] *= 0.3 -> 0.1800 | f = [0.18, 0.54, 0.28]
  đồng 3 (p=0.5): f[2] = 0.2800*0.5 + 0.5400*0.5 = 0.4100
  đồng 3 (p=0.5): f[1] = 0.5400*0.5 + 0.1800*0.5 = 0.3600
  đồng 3: f[0] *= 0.5 -> 0.0900 | f = [0.09, 0.36, 0.41]
Kết quả: 0.41


## 9. Code nộp LeetCode

In [13]:
class Solution:
    def probabilityOfHeads(self, prob: List[float], target: int) -> float:
        f = [0.0] * (target + 1)
        f[0] = 1.0
        for p in prob:
            q = 1.0 - p
            for j in range(target, 0, -1):
                f[j] = f[j] * q + f[j - 1] * p
            f[0] *= q
        return f[target]


sol = Solution()
for prob, target, want in examples:
    got = sol.probabilityOfHeads(prob, target)
    print(f"{got:.5f}")
    assert abs(got - want) < 1e-9

0.40000
0.03125


## 10. Vì sao đúng

**Bất biến:** sau khi xử lý `i` đồng đầu, `f[j]` (với `0 ≤ j ≤ target`) bằng xác suất có đúng `j` ngửa trong `i` đồng đó.

- **Cơ sở** (`i = 0`): `f = [1, 0, ..., 0]` — chưa tung đồng nào thì chắc chắn 0 ngửa.
- **Bước:** biến cố "có `j` ngửa trong `i` đồng" là hợp **rời nhau** của "(có `j` ngửa trong `i−1` đồng) **và** (đồng `i` sấp)" và "(có `j−1` ngửa trong `i−1` đồng) **và** (đồng `i` ngửa)". Do độc lập, mỗi phần là tích; do rời nhau, cộng lại → đúng công thức.
- **Thứ tự giảm dần:** khi tính `f[j]` mới, vị trí `j` và `j−1` vẫn chứa giá trị hàng `i−1` vì ta chỉ mới ghi đè các chỉ số `> j`. Cuối cùng `f[0]` mới chỉ phụ thuộc `f[0]` cũ.
- **Cắt ở `target`:** `f[j]` với `j ≤ target` không bao giờ cần tới `f[j']` với `j' > target`, nên bỏ các ô đó không ảnh hưởng.

Kết thúc, `f[target]` là đáp án.

## 11. Time / Space Complexity

Gọi `n = len(prob)`, `t = target`.

| Cách | Time | Space |
|---|---|---|
| 1. Liệt kê | `O(2^n · n)` | `O(n)` |
| 2. Đệ quy có nhớ | `O(n · t)` | `O(n · t)` + đệ quy sâu `n` |
| 3. DP 2 chiều | `O(n · t)` | `O(n · t)` |
| 4. DP 1 chiều | `O(n · t)` | `O(t)` |
| 5. Nhân đa thức | `O(n²)` | `O(n)` |

Với `n = t = 1000`: `10^6` phép tính — chạy cỡ vài trăm mili-giây trong Python.

## 12. Bộ test

### 12.1 Ví dụ đề + edge case

In [14]:
methods = {
    "memo": heads_memo,
    "dp2d": heads_dp2d,
    "dp1d": heads_dp1d,
    "poly": heads_poly,
    "submit": lambda p, t: Solution().probabilityOfHeads(p, t),
}
edge = [
    ([0.0, 0.0], 0, 1.0),            # không bao giờ ngửa
    ([0.0, 0.0], 1, 0.0),
    ([1.0, 1.0, 1.0], 3, 1.0),        # luôn ngửa
    ([1.0, 1.0, 1.0], 2, 0.0),
    ([0.3], 0, 0.7),
    ([0.5] * 4, 2, 6 / 16),           # nhị thức C(4,2)/16
    ([0.2, 0.9], 1, 0.2 * 0.1 + 0.8 * 0.9),
]
for name, f in methods.items():
    for prob, t, want in examples + edge:
        assert abs(f(prob, t) - want) < 1e-9, (name, prob, t)
print("Mọi cách đúng trên ví dụ + edge case")

Mọi cách đúng trên ví dụ + edge case


### 12.2 Đối chiếu ngẫu nhiên với brute force (seed cố định)

Kiểm tra hai tầng: (a) so với brute force số thực, (b) DP chạy trên `Fraction` cho kết quả **trùng tuyệt đối** với brute force chính xác.

In [15]:
def heads_dp1d_generic(prob, target, one):
    f = [one * 0] * (target + 1)
    f[0] = one
    for p in prob:
        for j in range(target, 0, -1):
            f[j] = f[j] * (1 - p) + f[j - 1] * p
        f[0] *= 1 - p
    return f[target]


rng = random.Random(1230)
for _ in range(300):
    n = rng.randint(1, 10)
    prob = [round(rng.random(), 2) for _ in range(n)]
    t = rng.randint(0, n)
    want = heads_brute(prob, t)
    for name, f in methods.items():
        assert abs(f(prob, t) - want) < 1e-9, (name, prob, t)
for _ in range(100):
    n = rng.randint(1, 8)
    probF = [Fraction(rng.randint(0, 10), 10) for _ in range(n)]
    t = rng.randint(0, n)
    assert heads_dp1d_generic(probF, t, Fraction(1)) == heads_brute_exact(probF, t)
print("400 test ngẫu nhiên: khớp brute force (số thực) và khớp tuyệt đối (Fraction)")

400 test ngẫu nhiên: khớp brute force (số thực) và khớp tuyệt đối (Fraction)


### 12.3 Test giới hạn `n = target = 1000`

Kiểm tra thêm: tổng phân phối bằng 1, và với mọi đồng `p = 0.5` thì kết quả trùng công thức nhị thức `C(n, k)/2^n`.

In [16]:
n = 1000
prob = [rng.random() for _ in range(n)]
for name in ["dp2d", "dp1d", "submit"]:
    t0 = time.perf_counter()
    r = methods[name](prob, 500)
    print(f"{name:7s} {r:.10f} trong {(time.perf_counter() - t0) * 1000:.0f} ms")
assert abs(heads_dp1d(prob, 500) - heads_dp2d(prob, 500)) < 1e-12
t0 = time.perf_counter()
prob_hi = [rng.uniform(0.998, 1.0) for _ in range(n)]
r_last = Solution().probabilityOfHeads(prob_hi, n)
print("target = n:", r_last, "≈ tích các p:", math.prod(prob_hi), f"({(time.perf_counter() - t0) * 1000:.0f} ms)")
assert math.isclose(r_last, math.prod(prob_hi), rel_tol=1e-9)
half = [0.5] * 60
dist_sum = sum(heads_poly(half, k) for k in range(61))
assert abs(dist_sum - 1) < 1e-9
assert all(abs(Solution().probabilityOfHeads(half, k) - math.comb(60, k) / 2 ** 60) < 1e-12 for k in range(61))
print("Tổng phân phối = 1, khớp công thức nhị thức khi mọi p = 0.5")

dp2d    0.0309896310 trong 61 ms


dp1d    0.0309896310 trong 32 ms


submit  0.0309896310 trong 22 ms


target = n: 0.38181802952739247 ≈ tích các p: 0.38181802952739247 (48 ms)
Tổng phân phối = 1, khớp công thức nhị thức khi mọi p = 0.5


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — DP một chiều nhưng duyệt `j` **tăng** dần

In [17]:
def wrong_increasing(prob, target):
    f = [0.0] * (target + 1)
    f[0] = 1.0
    for p in prob:
        f[0] *= 1 - p                            # cập nhật f[0] trước...
        for j in range(1, target + 1):           # ...rồi tăng dần: f[j-1] đã là giá trị MỚI
            f[j] = f[j] * (1 - p) + f[j - 1] * p
    return f[target]


prob = [0.4, 0.7, 0.5]
print("sai :", round(wrong_increasing(prob, 2), 4))
print("đúng:", round(heads_dp1d(prob, 2), 4))

sai : 0.1557
đúng: 0.41


**Vì sao sai:** `f[j−1]` đã được cập nhật cho đồng **hiện tại** nên công thức như thể đồng này được tung **nhiều lần** (một đồng góp nhiều mặt ngửa). **Sửa:** duyệt `j` giảm dần và cập nhật `f[0]` sau cùng (hoặc dùng mảng mới mỗi vòng).

### Lỗi 2 — Dùng công thức nhị thức với xác suất trung bình

In [18]:
def wrong_binomial(prob, target):
    n = len(prob)
    p = sum(prob) / n                            # "trung bình hoá" — sai!
    return math.comb(n, target) * p ** target * (1 - p) ** (n - target)


prob = [0.0, 1.0]
print("sai :", wrong_binomial(prob, 1))          # 0.5
print("đúng:", heads_dp1d(prob, 1))              # 1.0 (chắc chắn đúng 1 ngửa)

sai : 0.5
đúng: 1.0


**Vì sao sai:** công thức `C(n,k)p^k(1−p)^(n−k)` chỉ đúng khi **mọi** đồng có **cùng** xác suất `p`. Với `[0, 1]`, chắc chắn đúng 1 ngửa, nhưng "trung bình 0.5" cho 0.5. **Sửa:** DP.

### Lỗi 3 — Quên cập nhật `f[0]` (xác suất "toàn sấp")

In [19]:
def wrong_forget_f0(prob, target):
    f = [0.0] * (target + 1)
    f[0] = 1.0
    for p in prob:
        for j in range(target, 0, -1):
            f[j] = f[j] * (1 - p) + f[j - 1] * p
        # quên: f[0] *= 1 - p
    return f[target]


print("sai :", wrong_forget_f0([0.5] * 5, 0), "|", round(wrong_forget_f0([0.4, 0.7, 0.5], 2), 4))
print("đúng:", heads_dp1d([0.5] * 5, 0), "|", round(heads_dp1d([0.4, 0.7, 0.5], 2), 4))

sai : 1.0 | 0.55
đúng: 0.03125 | 0.41


**Vì sao sai:** vòng `for j in range(target, 0, -1)` không chạm `j = 0`. `f[0]` mãi bằng 1 → mọi `f[j]` bị thổi phồng. **Sửa:** `f[0] *= 1 - p` sau vòng trong.

### Lỗi 4 — Đệ quy có nhớ với `n = 1000` mà không lo độ sâu

In [20]:
def memo_depth_demo(prob, target):
    @lru_cache(maxsize=None)
    def go(i, k):
        if i == len(prob):
            return 1.0 if k == 0 else 0.0
        if k < 0:
            return 0.0
        return prob[i] * go(i + 1, k - 1) + (1 - prob[i]) * go(i + 1, k)
    return go(0, target)


try:
    print(memo_depth_demo([0.001] * 3000, 1))
except RecursionError as e:
    print("RecursionError với 3000 đồng:", type(e).__name__)
print("bản lặp:", Solution().probabilityOfHeads([0.001] * 3000, 1))

RecursionError với 3000 đồng: RecursionError
bản lặp: 0.14928646846257115


**Vì sao sai:** mỗi tầng đệ quy ứng với một đồng xu → độ sâu bằng `n`. Python giới hạn ~1000 tầng (LeetCode `n ≤ 1000` sát nút). **Sửa:** DP bottom-up (Cách 3/4) hoặc `sys.setrecursionlimit`.

## 14. Mẹo & nhận dạng pattern

- "Xác suất **đúng `k`** sự kiện độc lập xảy ra, mỗi sự kiện có xác suất riêng" → DP `f[i][j]` (Poisson binomial).
- Mỗi phần tử "chọn hoặc không chọn" và trạng thái là "đã chọn bao nhiêu" → **giống knapsack 0/1** → mảng 1 chiều duyệt **giảm dần**.
- Tích các đa thức bậc 1 `(q_i + p_i x)` ⇔ DP này. Gặp "đếm số cách / xác suất theo tổng" hãy nghĩ tới **hàm sinh**.
- Luôn kiểm tra: tổng phân phối bằng 1; trường hợp mọi `p` bằng nhau khớp nhị thức.
- Xác suất → dùng `Fraction` khi kiểm chứng để loại bỏ sai số làm tròn.

## 15. Biến thể

1. **Xác suất có ít nhất `k` ngửa:** `sum(f[k:])` (khi đó không được cắt ở `target`, cần cả mảng tới `n`).
2. **Kỳ vọng số mặt ngửa:** tuyến tính kỳ vọng → `sum(prob)`, không cần DP.
3. **Mỗi "đồng xu" có nhiều mặt** (xúc xắc với phân phối riêng), hỏi xác suất tổng bằng `S` → DP theo tổng, nhân đa thức bậc cao hơn.
4. **Đếm số cách** thay vì xác suất (mỗi đồng là "chọn/không chọn") → `f[j] += f[j−1]` → ra hệ số nhị thức.
5. **Dice Rolls (LeetCode 1155 — Number of Dice Rolls With Target Sum):** cùng khung DP theo tổng.

In [21]:
def at_least_k(prob, k):
    """Biến thể 1: P(số ngửa >= k)."""
    n = len(prob)
    f = [0.0] * (n + 1)
    f[0] = 1.0
    for p in prob:
        for j in range(n, 0, -1):
            f[j] = f[j] * (1 - p) + f[j - 1] * p
        f[0] *= 1 - p
    return sum(f[k:])


prob = [0.4, 0.7, 0.5]
print("P(>=2) =", round(at_least_k(prob, 2), 4), "= 0.41 + 0.14")
print("Kỳ vọng số ngửa =", sum(prob))

P(>=2) = 0.55 = 0.41 + 0.14
Kỳ vọng số ngửa = 1.6


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Liệt kê | mọi kết quả `2^n` | `O(2^n·n)` | `O(n)` | trọng tài |
| 2. Đệ quy có nhớ | `go(i, k)` | `O(n·t)` | `O(n·t)` | dễ nghĩ, cẩn thận độ sâu |
| 3. DP 2 chiều | bảng `f[i][j]` | `O(n·t)` | `O(n·t)` | dễ trace, dễ debug |
| 4. DP 1 chiều | `j` giảm dần | `O(n·t)` | `O(t)` | **nộp bài** |
| 5. Nhân đa thức | `Π(q_i + p_i x)` | `O(n²)` | `O(n)` | cần cả phân phối |

## 17. Bài tập vận dụng

Các cell TODO **chưa được gọi**: tự điền rồi bỏ comment dòng test.

### Bài 1 (đoán output)

`Solution().probabilityOfHeads([1.0, 0.0, 1.0, 0.0], 2)` = ?

### Bài 2 (tính tay)

`prob = [0.5, 0.2]`, `target = 1`. Lập bảng `f[i][j]` 3 hàng.

### Bài 3 (điền TODO) — toàn bộ phân phối bằng DP 1 chiều

In [22]:
def distribution(prob):
    """Trả về list d với d[k] = P(đúng k ngửa), k = 0..n."""
    n = len(prob)
    f = [0.0] * (n + 1)
    f[0] = 1.0
    for p in prob:
        # TODO: cập nhật f giảm dần như Cách 4 nhưng tới n
        raise NotImplementedError
    return f

# print([round(v, 4) for v in distribution([0.4, 0.7, 0.5])])   # [0.09, 0.36, 0.41, 0.14]

### Bài 4 (tự viết lại)

Viết lại Cách 3 nhưng dùng **hai mảng** `prev`, `cur` (không cần duyệt giảm dần). Vì sao khi có hai mảng thì thứ tự duyệt `j` không quan trọng?

In [23]:
def heads_two_rows_todo(prob, target):
    # TODO
    raise NotImplementedError

# print(round(heads_two_rows_todo([0.4, 0.7, 0.5], 2), 4))   # 0.41

### Bài 5 (mini challenge) — tổng điểm xúc xắc lệch

Có các xúc xắc, xúc xắc `i` cho bởi list `faces[i]` gồm xác suất ra mặt `1..len(faces[i])`. Tính xác suất tổng điểm đúng bằng `S`.

In [24]:
def dice_sum_todo(faces, S):
    # TODO: f[s] = P(tổng = s) sau các xúc xắc đã xét
    raise NotImplementedError

# print(dice_sum_todo([[0.5, 0.5], [0.5, 0.5]], 3))   # 0.5  (1+2 hoặc 2+1)

## 18. Gợi ý 3 mức

**Bài 1.** Mức 1: đồng `p = 1` luôn ngửa, `p = 0` luôn sấp. Mức 2: chắc chắn đúng 2 ngửa. Mức 3: `1.0`.

**Bài 2.** Mức 1: hàng 0: `[1, 0]`. Mức 2: hàng 1 (p=0.5): `[0.5, 0.5]`. Mức 3: hàng 2 (p=0.2): `f[1] = 0.5·0.8 + 0.5·0.2 = 0.5`.

**Bài 3.** Mức 1: vòng `for j in range(n, 0, -1)`. Mức 2: công thức y hệt Cách 4. Mức 3: `f[0] *= 1 − p` sau vòng trong.

**Bài 4.** Mức 1: `cur = [0.0] * (target + 1)` mỗi vòng. Mức 2: `cur[j] = prev[j]*(1−p) + (prev[j−1]*p if j else 0)`. Mức 3: `prev = cur`; thứ tự `j` không quan trọng vì chỉ đọc `prev` (không bao giờ bị ghi).

**Bài 5.** Mức 1: `f = [1.0]` (tổng 0 với xác suất 1). Mức 2: với mỗi xúc xắc, `g[s + v] += f[s] * faces[i][v − 1]`. Mức 3: trả `f[S]` nếu `S < len(f)` ngược lại `0`.

## 19. Đáp án tham khảo

In [25]:
# Bài 1, 2
r1 = Solution().probabilityOfHeads([1.0, 0.0, 1.0, 0.0], 2)
print(r1)
assert r1 == 1.0
r2 = heads_dp2d([0.5, 0.2], 1)
print(r2)
assert abs(r2 - 0.5) < 1e-12

1.0
0.5


In [26]:
def distribution(prob):
    n = len(prob)
    f = [0.0] * (n + 1)
    f[0] = 1.0
    for p in prob:
        for j in range(n, 0, -1):
            f[j] = f[j] * (1 - p) + f[j - 1] * p
        f[0] *= 1 - p
    return f


def heads_two_rows(prob, target):
    prev = [0.0] * (target + 1)
    prev[0] = 1.0
    for p in prob:
        cur = [0.0] * (target + 1)
        for j in range(target + 1):
            cur[j] = prev[j] * (1 - p) + (prev[j - 1] * p if j else 0.0)
        prev = cur
    return prev[target]


def dice_sum(faces, S):
    f = [1.0]
    for probs in faces:
        g = [0.0] * (len(f) + len(probs))
        for s, ps in enumerate(f):
            if ps:
                for v, pv in enumerate(probs, 1):
                    g[s + v] += ps * pv
        f = g
    return f[S] if 0 <= S < len(f) else 0.0


def dice_brute(faces, S):
    total = 0.0
    for combo in product(*[range(1, len(fc) + 1) for fc in faces]):
        if sum(combo) == S:
            pr = 1.0
            for fc, v in zip(faces, combo):
                pr *= fc[v - 1]
            total += pr
    return total


assert [round(v, 4) for v in distribution([0.4, 0.7, 0.5])] == [0.09, 0.36, 0.41, 0.14]
assert abs(heads_two_rows([0.4, 0.7, 0.5], 2) - 0.41) < 1e-12
assert abs(dice_sum([[0.5, 0.5], [0.5, 0.5]], 3) - 0.5) < 1e-12
rng = random.Random(8)
for _ in range(200):
    n = rng.randint(1, 8)
    prob = [rng.random() for _ in range(n)]
    d = distribution(prob)
    assert abs(sum(d) - 1) < 1e-9
    for k in range(n + 1):
        assert abs(d[k] - heads_brute(prob, k)) < 1e-9
        assert abs(heads_two_rows(prob, k) - d[k]) < 1e-9
    faces = []
    for _ in range(rng.randint(1, 4)):
        w = [rng.random() for _ in range(rng.randint(1, 4))]
        faces.append([x / sum(w) for x in w])
    S = rng.randint(0, 17)
    assert abs(dice_sum(faces, S) - dice_brute(faces, S)) < 1e-9
print("Đáp án bài 3, 4, 5 đúng")

Đáp án bài 3, 4, 5 đúng


## 20. Template nhớ nhanh + Checklist trước khi submit

```python
f = [0.0] * (target + 1); f[0] = 1.0
for p in prob:
    q = 1 - p
    for j in range(target, 0, -1):     # GIẢM dần (knapsack 0/1)
        f[j] = f[j] * q + f[j - 1] * p
    f[0] *= q                          # đừng quên j = 0
return f[target]
```

**Checklist:**
- [ ] `f[0] = 1` lúc đầu (chưa tung đồng nào → 0 ngửa chắc chắn)?
- [ ] Duyệt `j` **giảm dần** khi dùng một mảng?
- [ ] Cập nhật `f[0] *= 1 − p` mỗi vòng?
- [ ] Không dùng công thức nhị thức (xác suất khác nhau)?
- [ ] Không đệ quy sâu `n = 1000`?
- [ ] Thử: `p = 0`, `p = 1`, `target = 0`, `target = n`?